# DX 704 Week 4 Project

This week's project will test the learning speed of linear contextual bandits compared to unoptimized approaches.
You will start with building a preference data set for evaluation, and then implement different variations of LinUCB and visualize how fast they learn the preferences.


The full project description, a template notebook and supporting code are available on GitHub: [Project 4 Materials](https://github.com/bu-cds-dx704/dx704-project-04).


## Example Code

You may find it helpful to refer to these GitHub repositories of Jupyter notebooks for example code.

* https://github.com/bu-cds-omds/dx601-examples
* https://github.com/bu-cds-omds/dx602-examples
* https://github.com/bu-cds-omds/dx603-examples
* https://github.com/bu-cds-omds/dx704-examples

Any calculations demonstrated in code examples or videos may be found in these notebooks, and you are allowed to copy this example code in your homework answers.

In [1]:
%pip install pandas numpy scikit-learn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.8/10.8 MB 35.5 MB/s  0:00:006m0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.7/16.7 MB 41.3 MB/s  0:00:006m0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.2/9.2 MB 38.7 MB/s  0:00:00m eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.3/35.3 MB 37.0 MB/s  0:00:006m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8/8 [scikit-learn] [scikit-learn]
Note: you may need to restart the kernel to use updated packages.


## Part 1: Collect Rating Data

The file "recipes.tsv" in this repository has information about 100 recipes.
Make a new file "ratings.tsv" with two columns, recipe_slug (from recipes.tsv) and rating.
Populate the rating column with values between 0 and 1 where 0 is the worst and 1 is the best.
You can assign these ratings however you want within that range, but try to make it reflect a consistent set of preferences.
These could be your preferences, or a persona of your choosing (e.g. chocolate lover, bacon-obsessed, or sweet tooth).
Make sure that there are at least 10 ratings of zero and at least 10 ratings of one.


Hint: You may find it more convenient to assign raw ratings from 1 to 5 and then remap them as follows.

`ratings["rating"] = (ratings["rating_raw"] - 1) * 0.25`

In [2]:
import pandas as pd
import numpy as np

recipes = pd.read_csv("recipes.tsv", sep="\t")
tags = pd.read_csv("recipe-tags.tsv", sep="\t")

tag_sets = tags.groupby("recipe_slug")["recipe_tag"].apply(set)

love = {"chocolate", "chocolatechip", "chocolatecroissant"}
hate = {"seafood", "spicy", "spicyfood", "numbing", "pickledvegetables", "spam", "oysters", "shrimp", "scallops"}
like = {"dessert", "sweet", "sweettreat", "pastry", "frenchpastry", "baking", "breakfast", "brunch", "fruit"}
meh = {"condiment", "salad", "sidedish", "relish", "sauce"}

def raw_rating(slug):
    t = tag_sets.get(slug, set())
    if t & love:
        return 5
    if t & hate:
        return 1
    if t & like:
        return 4
    if t & meh:
        return 2
    return 3

ratings = recipes[["recipe_slug"]].copy()
ratings["rating_raw"] = ratings["recipe_slug"].apply(raw_rating)
ratings["rating"] = (ratings["rating_raw"] - 1) * 0.25

print((ratings["rating"] == 0).sum(), (ratings["rating"] == 1).sum())
assert (ratings["rating"] == 0).sum() >= 10
assert (ratings["rating"] == 1).sum() >= 10

ratings[["recipe_slug", "rating"]].to_csv("ratings.tsv", sep="\t", index=False)
ratings = pd.read_csv("ratings.tsv", sep="\t")
ratings.head()

14 11


,recipe_slug,rating
0,falafel,0.50
1,spamburger,0.00
2,bacon-fried-rice,0.75
3,chicken-fingers,0.50
4,apple-crisp,0.75


Submit "ratings.tsv" in Gradescope.

## Part 2: Construct Model Input

Use your file "ratings.tsv" combined with "recipe-tags.tsv" to create a new file "features.tsv" with a column recipe_slug, a column bias which is hard-coded to one, and a column for each tag that appears in "recipe-tags.tsv".
The tag column in this file should be a 0-1 encoding of the recipe tags for each recipe.
[Pandas reshaping function methods](https://pandas.pydata.org/docs/user_guide/reshaping.html) may be helpful.

The bias column will make later LinUCB calculations easier since it will just be another dimension.

Hint: For later modeling steps, it will be important to have the feature data (inputs) and the rating data (target outputs) in the same order.
It is highly recommended to make sure that "features.tsv" and "ratings.tsv" have the recipe slugs in the same order.

In [3]:
# YOUR CHANGES HERE

tags["value"] = 1
wide = tags.pivot_table(index="recipe_slug", columns="recipe_tag", values="value", aggfunc="max", fill_value=0)
wide.columns.name = None

wide = wide.reindex(ratings["recipe_slug"], fill_value=0)
wide.index.name = "recipe_slug"

features = wide.reset_index()
features.insert(1, "bias", 1)
features = features.astype({c: int for c in features.columns if c != "recipe_slug"})

assert (features["recipe_slug"].values == ratings["recipe_slug"].values).all()

features.to_csv("features.tsv", sep="\t", index=False)
features.head()

,recipe_slug,bias,alfredo,almond,american,appetizer,appetizers,apple,asiancuisine,asparagus,...,udonnoodles,vanilla,vanillaicecream,vegan,vegetables,vegetarian,warm,whippedcream,winter,yeastdough
0,falafel,1,0,0,0,1,0,0,0,0,...,0,0,0,1,0,1,0,0,0,0
1,spamburger,1,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,bacon-fried-rice,1,0,0,0,0,0,0,0,0,...,0,0,0,0,1,0,0,0,0,0
3,chicken-fingers,1,0,0,0,1,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,apple-crisp,1,0,0,0,0,0,1,0,0,...,0,0,0,0,0,0,0,0,1,0


Submit "features.tsv" in Gradescope.

## Part 3: Linear Preference Model

Use your feature and rating files to build a ridge regression model with ridge regression's regularization parameter $\alpha$ set to 1.


Hint: If you are using scikit-learn modeling classes, you should use `fit_intercept=False` since that intercept value will be redundant with the bias coefficient.

Hint: The estimate component of the bounds should match the previous estimate, so you should be able to just focus on the variance component of the bounds now.

In [4]:
# YOUR CHANGES HERE

from sklearn.linear_model import Ridge

X = features.drop(columns=["recipe_slug"]).values.astype(float)
y = ratings["rating"].values.astype(float)
feature_names = features.drop(columns=["recipe_slug"]).columns

model = Ridge(alpha=1.0, fit_intercept=False)
model.fit(X, y)

,"fit_intercept fit_intercept: bool, default=TrueWhether to fit the intercept for this model. If setto false, no intercept will be used in calculations(i.e. ``X`` and ``y`` are expected to be centered).",False
,"alpha alpha: float or array-like of shape (n_targets,), default=1.0Constant that multiplies the L2 term, controlling regularizationstrength. `alpha` must be a non-negative float i.e. in `[0, inf)`.When `alpha = 0`, the objective is equivalent to ordinary leastsquares, solved by the :class:`LinearRegression` object. For numericalreasons, using `alpha = 0` with the `Ridge` object is not advised.Instead, you should use the :class:`LinearRegression` object.If an array is passed, penalties are assumed to be specific to thetargets. Hence they must correspond in number.See :ref:`sphx_glr_auto_examples_linear_model_plot_ridge_coeffs.py`for an illustration of the effect of alpha on the model coefficients.",1.0
,"copy_X copy_X: bool, default=TrueIf True, X will be copied; else, it may be overwritten.",True
,"max_iter max_iter: int, default=NoneMaximum number of iterations for conjugate gradient solver.For 'sparse_cg' and 'lsqr' solvers, the default value is determinedby scipy.sparse.linalg. For 'sag' solver, the default value is 1000.For 'lbfgs' solver, the default value is 15000.",None
,"tol tol: float, default=1e-4The precision of the solution (`coef_`) is determined by `tol` whichspecifies a different convergence criterion for each solver:- 'svd': `tol` has no impact.- 'cholesky': `tol` has no impact.- 'sparse_cg': norm of residuals smaller than `tol`.- 'lsqr': `tol` is set as atol and btol of scipy.sparse.linalg.lsqr, which control the norm of the residual vector in terms of the norms of matrix and coefficients.- 'sag' and 'saga': relative change of coef smaller than `tol`.- 'lbfgs': maximum of the absolute (projected) gradient=max|residuals| smaller than `tol`... versionchanged:: 1.2 Default value changed from 1e-3 to 1e-4 for consistency with other linear models.",0.0001
,"solver solver: {'auto', 'svd', 'cholesky', 'lsqr', 'sparse_cg', 'sag', 'saga', 'lbfgs'}, default='auto'Solver to use in the computational routines:- 'auto' chooses the solver automatically based on the type of data.- 'svd' uses a Singular Value Decomposition of X to compute the Ridge coefficients. It is the most stable solver, in particular more stable for singular matrices than 'cholesky' at the cost of being slower.- 'cholesky' uses the standard :func:`scipy.linalg.solve` function to obtain a closed-form solution.- 'sparse_cg' uses the conjugate gradient solver as found in :func:`scipy.sparse.linalg.cg`. As an iterative algorithm, this solver is more appropriate than 'cholesky' for large-scale data (possibility to set `tol` and `max_iter`).- 'lsqr' uses the dedicated regularized least-squares routine :func:`scipy.sparse.linalg.lsqr`. It is the fastest and uses an iterative procedure.- 'sag' uses a Stochastic Average Gradient descent, and 'saga' uses its improved, unbiased version named SAGA. Both methods also use an iterative procedure, and are often faster than other solvers when both n_samples and n_features are large. Note that 'sag' and 'saga' fast convergence is only guaranteed on features with approximately the same scale. You can preprocess the data with a scaler from :mod:`sklearn.preprocessing`.- 'lbfgs' uses L-BFGS-B algorithm implemented in :func:`scipy.optimize.minimize`. It can be used only when `positive` is True.All solvers except 'svd' support both dense and sparse data. However, only'lsqr', 'sag', 'sparse_cg', and 'lbfgs' support sparse input when`fit_intercept` is True... versionadded:: 0.17 Stochastic Average Gradient descent solver... versionadded:: 0.19 SAGA solver.",'auto'
,"positive positive: bool, default=FalseWhen set to ``True``, forces the coefficients to be positive.Only 'lbfgs' solver is supported in this case.",False
,"random_state random_state: int, RandomState instance, default=NoneUsed when ``solver`` == 'sag' or 'saga'

Save the coefficients of this model in a file "model.tsv" with columns "recipe_tag" and "coefficient".
Do not add anything for the `intercept_` attribute of a scikit-learn model; this will be covered by the coefficient for the bias column added in part 2.

In [5]:
# YOUR CHANGES HERE

model_df = pd.DataFrame({"recipe_tag": feature_names, "coefficient": model.coef_})
model_df.to_csv("model.tsv", sep="\t", index=False)
model_df.sort_values("coefficient", ascending=False).head(10)

,recipe_tag,coefficient
0,bias,0.359789
88,dessert,0.235735
51,chocolate,0.176853
14,baking,0.120515
27,breakfast,0.111982
67,cookies,0.103018
227,savory,0.084723
33,brunch,0.082109
141,holiday,0.077128
108,frenchpastry,0.074879


Submit "model.tsv" in Gradescope.

## Part 4: Recipe Estimates

Use the recipe model to estimate the score of every recipe.
Save these estimates to a file "estimates.tsv" with columns recipe_slug and score_estimate.

In [6]:
# YOUR CHANGES HERE

estimates = pd.DataFrame({
    "recipe_slug": features["recipe_slug"],
    "score_estimate": X @ model.coef_
})
estimates.to_csv("estimates.tsv", sep="\t", index=False)
estimates.head()

,recipe_slug,score_estimate
0,falafel,0.453472
1,spamburger,0.041544
2,bacon-fried-rice,0.740884
3,chicken-fingers,0.455937
4,apple-crisp,0.726305


Submit "estimates.tsv" in Gradescope.

## Part 5: LinUCB Bounds

Calculate the upper bounds of LinUCB using data corresponding to trying every recipe once and receiving the rating in "ratings.tsv" as the reward.
Keep the ridge regression regularization parameter at 1, and set LinUCB's $\alpha$ parameter to 2.
Save these upper bounds to a file "bounds.tsv" with columns recipe_slug and score_bound.

In [7]:
# YOUR CHANGES HERE

ridge_lambda = 1.0
linucb_alpha = 2.0
d = X.shape[1]

A = X.T @ X + ridge_lambda * np.eye(d)
A_inv = np.linalg.inv(A)
b = X.T @ y
theta = A_inv @ b

variance = np.einsum("ij,jk,ik->i", X, A_inv, X)
score_bound = X @ theta + linucb_alpha * np.sqrt(variance)

assert np.allclose(X @ theta, estimates["score_estimate"].values)

bounds = pd.DataFrame({
    "recipe_slug": features["recipe_slug"],
    "score_bound": score_bound
})
bounds.to_csv("bounds.tsv", sep="\t", index=False)
bounds.head()

,recipe_slug,score_bound
0,falafel,2.219727
1,spamburger,1.935880
2,bacon-fried-rice,2.607041
3,chicken-fingers,2.250053
4,apple-crisp,2.502646


Submit "bounds.tsv" in Gradescope.

## Part 6: Make Online Recommendations

Implement LinUCB to make 100 recommendations starting with no data and using the same parameters as in part 5.
One recommendation should be made at a time and you can break ties arbitrarily.
After each recommendation, use the rating from part 1 as the reward to update the LinUCB data.
Record the recommendations made in a file "recommendations.tsv" with columns "recipe_slug", "score_bound", and "reward".
The rows in this file should be in the same order as the recommendations were made.

Hint: do not remove recipes after each recommendation.
Repeating recommendations is expected.

In [8]:
# YOUR CHANGES HERE

A = ridge_lambda * np.eye(d)
b = np.zeros(d)
slugs = features["recipe_slug"].values
rows = []

for step in range(100):
    A_inv = np.linalg.inv(A)
    theta = A_inv @ b
    variance = np.einsum("ij,jk,ik->i", X, A_inv, X)
    ucb = X @ theta + linucb_alpha * np.sqrt(variance)
    idx = int(np.argmax(ucb))
    reward = y[idx]
    x = X[idx]
    A += np.outer(x, x)
    b += reward * x
    rows.append({"recipe_slug": slugs[idx], "score_bound": ucb[idx], "reward": reward})

recommendations = pd.DataFrame(rows)
recommendations.to_csv("recommendations.tsv", sep="\t", index=False)
recommendations.head(20)

,recipe_slug,score_bound,reward
0,apple-crumble,7.483315,0.75
1,ma-la-chicken,7.242589,0.00
2,quesadillas,7.160666,0.50
3,ramen,7.172298,0.50
4,chocolate-babka,6.954279,1.00
5,pain-au-chocolat,7.003631,1.00
6,spamburger,7.001091,0.00
7,bacon-fried-rice,6.932329,0.75
8,nacho-fries,6.713657,0.50
9,cranberry-sauce,6.712202,0.75


Submit "recommendations.tsv" in Gradescope.

## Part 7: Acknowledgments

Make a file "acknowledgments.txt" documenting any outside sources or help on this project.
If you discussed this assignment with anyone, please acknowledge them here.
If you used any libraries not mentioned in this module's content, please list them with a brief explanation what you used them for.
If you used any generative AI tools, please add links to your transcripts below, and any other information that you feel is necessary to comply with the generative AI policy.
If no acknowledgements are appropriate, just write none in the file.


Submit "acknowledgments.txt" in Gradescope.

## Part 8: Code

Please submit a Jupyter notebook that can reproduce all your calculations and recreate the previously submitted files.


Submit "project.ipynb" in Gradescope.